# Ensemble SVM RBF + XGBoost + CNN spatial head

## Abstract

Questo notebook combina esclusivamente SVM RBF, XGBoost e CNN spatial head tramite soft voting. Carica i migliori artefatti e le predizioni dei modelli base senza riaddestrarli, verifica split, mapping delle classi, campioni e versioni, calibra i logit CNN con temperature scaling appreso sul validation e seleziona pesi e soglia solo sul validation. Usa NumPy, pandas, scikit-learn, SciPy, PyTorch e joblib per valutare l'ensemble, confrontarlo con ConvNeXt sul test e produrre un bundle portabile.

## Schema della pipeline completa

1. Setup locale/Colab e definizione delle directory persistenti.
2. Caricamento dell'indice e dello split comune, con hash stabile.
3. Risoluzione degli artefatti migliori di SVM, XGBoost, CNN e ConvNeXt; ConvNeXt resta fuori dall'ensemble.
4. Verifica di mapping, hash, copertura, ordine e versioni delle predizioni.
5. Calibrazione sul validation: probabilità SVM/XGBoost già calibrate dai rispettivi notebook; temperature scaling per i logit CNN.
6. Soft voting uniforme e ricerca a griglia dei pesi non negativi, con soglia ottimizzata esclusivamente sul validation.
7. Valutazione train/validation/test, diversità, bootstrap appaiato, McNemar e confronto con ConvNeXt.
8. Esportazione del bundle: pipeline SVM, modello e calibratore XGBoost, CNN TorchScript e configurazione dell'ensemble.

## Schema visivo della pipeline dell'ensemble e dei dati

Feature map e indice condivisi
→ SVM RBF: pooling tabulare salvato → probabilità calibrata
→ XGBoost: pooling tabulare salvato → probabilità calibrata
→ CNN spatial head: mappe normalizzate per canale → logit → temperature scaling
→ soft voting uniforme e pesato
→ soglia bloccata dal validation
→ predizioni e metriche sul test

In parallelo, ConvNeXt usa le stesse mappe normalizzate e lo stesso split, ma le sue predizioni entrano solo nel confronto finale: non riceve alcun peso dell'ensemble.

## 1. Setup portabile dei percorsi

Questa cella usa la stessa configurazione locale/Colab degli altri notebook. In Colab monta Drive e usa la root persistente del progetto; l'ensemble legge e scrive soltanto tramite le directory derivate.

In [1]:
from pathlib import Path
import importlib.util

try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
if IS_COLAB:
    from google.colab import drive
    DRIVE_MOUNT = Path("/content/drive")
    if not (DRIVE_MOUNT / "MyDrive").exists():
        drive.mount(str(DRIVE_MOUNT), force_remount=False)
    PROJECT_ROOT = DRIVE_MOUNT / "MyDrive" / "Magistrale" / "Advanced_ML" / "progetto_aml"
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if not (PROJECT_ROOT / "dataset").is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent
PERSISTENT_DATASET_DIR = PROJECT_ROOT / "dataset"
DATASET_DIR = PERSISTENT_DATASET_DIR
PROCESSED_DATASET_DIR = PERSISTENT_DATASET_DIR
ARTIFACTS_DIR, RESULTS_DIR = PROJECT_ROOT / "artifacts", PROJECT_ROOT / "results"
if not PERSISTENT_DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory non trovata: {PERSISTENT_DATASET_DIR}")
for directory in (ARTIFACTS_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("Ambiente:", "Google Colab" if IS_COLAB else "Locale")
print("Risultati:", RESULTS_DIR)

Ambiente: Locale
Risultati: C:\Users\leona\venv_aml\results


## 2. Dipendenze, configurazione e metriche comuni

Questa cella installa soltanto pacchetti assenti, imposta il seed per tutte le procedure casuali e importa le funzioni per calibrazione, bootstrap, metriche e bundle. I run ID opzionali consentono di fissare un esperimento specifico; se sono null viene scelto il più recente run completo.

In [2]:
import hashlib
import importlib
import json
import platform
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

required = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "sklearn": "scikit-learn", "scipy": "scipy", "joblib": "joblib"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize_scalar
from scipy.stats import binomtest
import sklearn
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, brier_score_loss, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

SEED, GRID_STEP, BOOTSTRAP_SAMPLES = 42, 0.05, 2000
MODEL_NAME = "ensemble"
COMPONENT_RUN_IDS = {"svm_rbf": None, "xgboost": None, "cnn_spatial_head": None, "convnext": None}
CLASS_MAPPING = {"features_0.npy": {"label": 0, "name": "real"}, "features_1.npy": {"label": 1, "name": "fake"}}
POSITIVE_LABEL, POSITIVE_CLASS = 1, "fake"
random.seed(SEED); np.random.seed(SEED)
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__, "scikit_learn": sklearn.__version__}
print(json.dumps(VERSIONS, indent=2))

{
  "python": "3.12.10",
  "numpy": "2.5.2",
  "pandas": "3.0.5",
  "scipy": "1.18.1",
  "scikit_learn": "1.9.1"
}


## 3. Indice, split condiviso e hash

Questa cella carica e verifica l'indice persistente e lo split comune. L'hash include nome e indici di ogni partizione, così può essere confrontato con ogni artefatto di modello e con le statistiche spaziali.

In [3]:
SAMPLE_INDEX_PATH, SPLIT_PATH = PROCESSED_DATASET_DIR / "sample_index.csv", ARTIFACTS_DIR / "splits_seed42.npz"
if not SAMPLE_INDEX_PATH.is_file() or not SPLIT_PATH.is_file():
    raise FileNotFoundError("sample_index.csv o splits_seed42.npz mancanti: eseguire prima dataset_exploration.ipynb")
sample_index = pd.read_csv(SAMPLE_INDEX_PATH)
required_columns = {"sample_id", "source_file", "source_index", "label"}
if not required_columns.issubset(sample_index.columns) or not sample_index["sample_id"].is_unique:
    raise ValueError("sample_index.csv non rispetta il contratto comune")
labels = sample_index["label"].to_numpy(dtype=np.int64)
if not np.array_equal(labels, sample_index["source_file"].map(lambda value: CLASS_MAPPING[value]["label"]).to_numpy()):
    raise ValueError("Mapping delle classi incompatibile con sample_index.csv")
splits = np.load(SPLIT_PATH, allow_pickle=False)
train_indices, validation_indices, test_indices = (splits[name].astype(np.int64) for name in ("train_indices", "validation_indices", "test_indices"))
merged = np.concatenate([train_indices, validation_indices, test_indices])
if len(np.unique(merged)) != len(sample_index) or not np.array_equal(np.sort(merged), np.arange(len(sample_index))):
    raise ValueError("Split non disgiunto o non esaustivo")
digest = hashlib.sha256()
for name, indices in (("train", train_indices), ("validation", validation_indices), ("test", test_indices)):
    digest.update(name.encode("utf-8")); digest.update(np.asarray(indices, dtype="<i8").tobytes())
SPLIT_HASH = digest.hexdigest()
expected_split = np.empty(len(sample_index), dtype=object)
expected_split[train_indices], expected_split[validation_indices], expected_split[test_indices] = "train", "validation", "test"
print("Split hash:", SPLIT_HASH)

Split hash: e23bd01e9ec079e665ed5cb58cdba6c190d40bb702919412e2f5f5aeca63cbd6


## 4. Risoluzione dei run e verifica degli artefatti richiesti

Questa cella individua il run completo più recente per ciascun modello, oppure usa un run ID esplicitamente configurato. Se CNN o ConvNeXt non sono stati ancora eseguiti, interrompe in modo esplicito senza creare risultati parziali dell'ensemble.

In [4]:
REQUIRED_FILES = {
    "svm_rbf": ("predictions.csv", "best_model.joblib"),
    "xgboost": ("predictions.csv", "best_model.json", "model_bundle.joblib"),
    "cnn_spatial_head": ("predictions.csv", "best_model_scripted.pt", "model_config.json"),
    "convnext": ("predictions.csv", "best_model_scripted.pt", "model_config.json"),
}
def resolve_run(model_name, requested_run_id, required_files):
    model_root = RESULTS_DIR / model_name
    if requested_run_id is not None:
        candidate = model_root / requested_run_id
        candidates = [candidate]
    elif model_root.is_dir():
        candidates = sorted((path for path in model_root.iterdir() if path.is_dir()), key=lambda path: path.stat().st_mtime, reverse=True)
    else:
        candidates = []
    for candidate in candidates:
        if all((candidate / filename).is_file() for filename in required_files):
            return candidate
    missing = ", ".join(required_files)
    raise FileNotFoundError(f"Nessun run completo per {model_name}. Richiesti: {missing}")

component_dirs = {
    name: resolve_run(name, COMPONENT_RUN_IDS[name], REQUIRED_FILES[name])
    for name in REQUIRED_FILES
}
for name, path in component_dirs.items():
    print(f"{name}: {path.name}")

svm_rbf: seed42_20260916T133639Z
xgboost: seed42_20260916T142429Z
cnn_spatial_head: seed42_20260917T082650Z
convnext: seed42_20260916T195419Z


## 5. Caricamento e allineamento delle predizioni

Questa cella carica configurazioni, bundle e predizioni. Verifica mapping, classe positiva, hash dello split, schema, assenza di valori mancanti e allineamento univoco di sample_id, file, indice, split e label tra tutti i modelli.

In [5]:
def balanced_error(*args, **kwargs):
    return "balanced_error", 0.0

svm_bundle = joblib.load(component_dirs["svm_rbf"] / "best_model.joblib")
xgb_bundle = joblib.load(component_dirs["xgboost"] / "model_bundle.joblib")
with (component_dirs["cnn_spatial_head"] / "model_config.json").open(encoding="utf-8") as handle:
    cnn_config = json.load(handle)
with (component_dirs["convnext"] / "model_config.json").open(encoding="utf-8") as handle:
    convnext_config = json.load(handle)
metadata = {"svm_rbf": svm_bundle, "xgboost": xgb_bundle, "cnn_spatial_head": cnn_config, "convnext": convnext_config}
for name, config in metadata.items():
    if config.get("class_mapping") != CLASS_MAPPING or config.get("positive_class") != POSITIVE_CLASS:
        raise ValueError(f"Mapping o classe positiva incompatibile nell'artefatto {name}")
    if config.get("split_hash") != SPLIT_HASH:
        raise ValueError(f"Split hash incompatibile nell'artefatto {name}")
    versions = config.get("versions", {})
    if not isinstance(versions, dict):
        raise ValueError(f"Versioni mancanti o non valide nell'artefatto {name}")

expected = sample_index[["sample_id", "source_file", "source_index", "label"]].copy()
expected["split"] = expected_split
prediction_columns = {"sample_id", "source_file", "source_index", "split", "y_true", "raw_score", "probability", "y_pred"}
component_predictions = {}
for name, directory in component_dirs.items():
    frame = pd.read_csv(directory / "predictions.csv")
    if not prediction_columns.issubset(frame.columns) or len(frame) != len(expected) or not frame["sample_id"].is_unique:
        raise ValueError(f"Schema o copertura predizioni non validi per {name}")
    frame = frame.set_index("sample_id").reindex(expected["sample_id"]).reset_index()
    for column in ("source_file", "source_index", "split"):
        if not frame[column].equals(expected[column]):
            raise ValueError(f"Predizioni {name} non allineate per {column}")
    if not np.array_equal(frame["y_true"].to_numpy(dtype=np.int64), expected["label"].to_numpy(dtype=np.int64)):
        raise ValueError(f"Predizioni {name} non allineate per y_true")
    if not np.isfinite(frame[["raw_score", "probability"]].to_numpy(dtype=float)).all():
        raise ValueError(f"Predizioni non finite per {name}")
    if ((frame["probability"] < 0) | (frame["probability"] > 1)).any():
        raise ValueError(f"Probabilità fuori intervallo per {name}")
    component_predictions[name] = frame
print("Predizioni convalidate e riallineate:", len(expected))

ValueError: Predizioni convnext non allineate per y_true

## 6. Calibrazione delle probabilità

Questa cella conserva le probabilità SVM e XGBoost già calibrate con Platt scaling nei notebook base. Applica temperature scaling soltanto ai logit CNN, ottimizzando la negative log-likelihood sul validation e senza consultare il test.

In [ ]:
ensemble_started = time.perf_counter()
def sigmoid(values):
    values = np.asarray(values, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-np.clip(values, -700, 700)))
def validation_mask(frame):
    return frame["split"].to_numpy() == "validation"
def fit_temperature(logits, targets):
    def objective(temperature):
        scaled = logits / temperature
        return np.mean(np.logaddexp(0.0, scaled) - targets * scaled)
    result = minimize_scalar(objective, bounds=(0.05, 10.0), method="bounded")
    if not result.success or not np.isfinite(result.x):
        raise RuntimeError("Temperature scaling non riuscito")
    return float(result.x)

for name in ("svm_rbf", "xgboost"):
    calibration = metadata[name].get("calibration", {})
    if "Platt" not in str(calibration.get("method", "")):
        raise ValueError(f"{name} non dichiara una calibrazione Platt compatibile")
cnn_frame = component_predictions["cnn_spatial_head"]
cnn_validation = validation_mask(cnn_frame)
CNN_TEMPERATURE = fit_temperature(cnn_frame.loc[cnn_validation, "raw_score"].to_numpy(float), cnn_frame.loc[cnn_validation, "y_true"].to_numpy(float))
component_probabilities = {
    "svm_rbf": component_predictions["svm_rbf"]["probability"].to_numpy(float),
    "xgboost": component_predictions["xgboost"]["probability"].to_numpy(float),
    "cnn_spatial_head": sigmoid(cnn_frame["raw_score"].to_numpy(float) / CNN_TEMPERATURE),
}
print(f"Temperatura CNN appresa sul validation: {CNN_TEMPERATURE:.5f}")

## 7. Selezione dei pesi e della soglia sul validation

Questa cella confronta soft voting uniforme e pesato. Esplora pesi con passo 0.05, non negativi e con somma uno; per ogni combinazione sceglie la soglia che massimizza la balanced accuracy soltanto sul validation. I valori finali vengono bloccati prima del test.

In [ ]:
y_all = expected["label"].to_numpy(dtype=np.int64)
validation_rows = expected["split"].to_numpy() == "validation"
def select_threshold(targets, probabilities):
    candidates = np.unique(np.concatenate([[0.0], probabilities, [0.5, 1.0]]))
    scores = np.array([balanced_accuracy_score(targets, probabilities >= candidate) for candidate in candidates])
    return float(candidates[np.argmax(scores)]), float(np.max(scores))
def combine(weights):
    return weights[0] * component_probabilities["svm_rbf"] + weights[1] * component_probabilities["xgboost"] + weights[2] * component_probabilities["cnn_spatial_head"]

UNIFORM_WEIGHTS = np.array([1 / 3, 1 / 3, 1 / 3], dtype=float)
uniform_probability = combine(UNIFORM_WEIGHTS)
UNIFORM_THRESHOLD, uniform_validation_score = select_threshold(y_all[validation_rows], uniform_probability[validation_rows])
grid_values = np.arange(0, 1 + GRID_STEP / 2, GRID_STEP)
best = {"score": -np.inf, "weights": None, "threshold": None}
for svm_weight in grid_values:
    for xgb_weight in grid_values:
        cnn_weight = 1.0 - svm_weight - xgb_weight
        if cnn_weight < -1e-12:
            continue
        weights = np.array([svm_weight, xgb_weight, max(0.0, cnn_weight)])
        probability = combine(weights)
        threshold, score = select_threshold(y_all[validation_rows], probability[validation_rows])
        if score > best["score"] + 1e-12:
            best = {"score": score, "weights": weights, "threshold": threshold}
WEIGHTED_WEIGHTS, WEIGHTED_THRESHOLD = best["weights"], best["threshold"]
weighted_probability = combine(WEIGHTED_WEIGHTS)
print("Uniforme:", UNIFORM_WEIGHTS, "soglia", UNIFORM_THRESHOLD, "BA validation", uniform_validation_score)
print("Pesato:", WEIGHTED_WEIGHTS, "soglia", WEIGHTED_THRESHOLD, "BA validation", best["score"])

## 8. Metriche, predizioni e analisi della diversità

Questa cella valuta modelli e ensemble su ogni split usando soglie congelate dal validation. Salva predizioni standardizzate dell'ensemble, metriche comuni, correlazioni delle probabilità e degli errori, accordo fra componenti e conteggi dei casi corretti solo da un modello.

In [ ]:
RUN_ID = f"seed{SEED}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
RUN_DIR, FIGURES_DIR = RESULTS_DIR / MODEL_NAME / RUN_ID, RESULTS_DIR / MODEL_NAME / RUN_ID / "figures"
RUN_DIR.mkdir(parents=True, exist_ok=False); FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def model_metrics(targets, probabilities, threshold):
    predicted = (probabilities >= threshold).astype(np.int64)
    matrix = confusion_matrix(targets, predicted, labels=[0, 1]); tn, fp, fn, tp = matrix.ravel()
    return {"accuracy": accuracy_score(targets, predicted), "balanced_accuracy": balanced_accuracy_score(targets, predicted), "precision": precision_score(targets, predicted, zero_division=0), "recall": recall_score(targets, predicted, zero_division=0), "specificity": tn / (tn + fp) if tn + fp else np.nan, "f1": f1_score(targets, predicted, zero_division=0), "roc_auc": roc_auc_score(targets, probabilities), "pr_auc": average_precision_score(targets, probabilities), "brier_score": brier_score_loss(targets, probabilities), "confusion_matrix": matrix}
def logit(probabilities):
    probabilities = np.clip(probabilities, 1e-7, 1 - 1e-7)
    return np.log(probabilities / (1 - probabilities))
def threshold_for_component(name, probabilities):
    return select_threshold(y_all[validation_rows], probabilities[validation_rows])[0]

evaluation_probabilities = {
    "svm_rbf": component_probabilities["svm_rbf"], "xgboost": component_probabilities["xgboost"],
    "cnn_spatial_head": component_probabilities["cnn_spatial_head"], "ensemble_uniform": uniform_probability,
    "ensemble_weighted": weighted_probability, "convnext": component_predictions["convnext"]["probability"].to_numpy(float),
}
evaluation_thresholds = {name: threshold_for_component(name, probabilities) for name, probabilities in evaluation_probabilities.items()}
evaluation_thresholds["ensemble_uniform"], evaluation_thresholds["ensemble_weighted"] = UNIFORM_THRESHOLD, WEIGHTED_THRESHOLD
metrics_rows, ensemble_frames = [], []
for model_name, probabilities in evaluation_probabilities.items():
    for split_name in ("train", "validation", "test"):
        mask = expected["split"].to_numpy() == split_name
        values = model_metrics(y_all[mask], probabilities[mask], evaluation_thresholds[model_name])
        metrics_rows.append({"run_id": RUN_ID, "model": model_name, "split": split_name, "n_samples": int(mask.sum()), **{key: value for key, value in values.items() if key != "confusion_matrix"}, "threshold": evaluation_thresholds[model_name]})
    if model_name.startswith("ensemble"):
        frame = expected.copy()
        frame["y_true"], frame["raw_score"], frame["probability"] = y_all, logit(probabilities), probabilities
        frame["y_pred"] = (probabilities >= evaluation_thresholds[model_name]).astype(np.int64)
        ensemble_frames.append((model_name, frame))
metrics_df = pd.DataFrame(metrics_rows)
ensemble_build_seconds = time.perf_counter() - ensemble_started
timing_cache = {}
for component_name, component_directory in component_dirs.items():
    timing_frame = pd.read_csv(component_directory / "metrics.csv")
    for _, timing_row in timing_frame.iterrows():
        timing_cache[(component_name, timing_row["split"])] = (
            float(timing_row.get("training_time_seconds", np.nan)),
            float(timing_row.get("inference_time_seconds", np.nan)),
        )
def timing_values(model_name, split_name):
    if model_name in component_dirs:
        return timing_cache.get((model_name, split_name), (np.nan, np.nan))
    component_times = [timing_values(component, split_name) for component in ("svm_rbf", "xgboost", "cnn_spatial_head")]
    return (
        float(sum(value[0] for value in component_times) + ensemble_build_seconds),
        float(sum(value[1] for value in component_times)),
    )
metrics_df[["training_time_seconds", "inference_time_seconds"]] = metrics_df.apply(
    lambda row: pd.Series(timing_values(row["model"], row["split"])), axis=1
)
metrics_df.to_csv(RUN_DIR / "metrics.csv", index=False)
for name, frame in ensemble_frames:
    frame.to_csv(RUN_DIR / f"predictions_{name}.csv", index=False)

test_mask = expected["split"].to_numpy() == "test"
component_names = ["svm_rbf", "xgboost", "cnn_spatial_head"]
test_probabilities = pd.DataFrame({name: component_probabilities[name][test_mask] for name in component_names})
test_predictions = pd.DataFrame({name: test_probabilities[name].to_numpy() >= evaluation_thresholds[name] for name in component_names})
test_errors = test_predictions.ne(y_all[test_mask], axis=0)
probability_correlation, error_correlation = test_probabilities.corr(), test_errors.astype(int).corr()
agreement_rows = [{"model_a": first, "model_b": second, "agreement": float((test_predictions[first] == test_predictions[second]).mean())} for index, first in enumerate(component_names) for second in component_names[index + 1:]]
unique_correct = {name: int(((test_predictions[name].to_numpy() == y_all[test_mask]) & (test_predictions.drop(columns=name).ne(y_all[test_mask], axis=0).all(axis=1).to_numpy())).sum()) for name in component_names}
probability_correlation.to_csv(RUN_DIR / "probability_correlation.csv")
error_correlation.to_csv(RUN_DIR / "error_correlation.csv")
pd.DataFrame(agreement_rows).to_csv(RUN_DIR / "agreement.csv", index=False)
pd.DataFrame([{"model": name, "unique_correct_cases": count} for name, count in unique_correct.items()]).to_csv(RUN_DIR / "unique_correct_cases.csv", index=False)
display(metrics_df.loc[metrics_df["split"] == "test"].round(4))

## 9. Figure dell'ensemble e della diversità

Questa cella salva curve ROC, precision-recall e calibrazione dell'ensemble pesato, oltre alle matrici di correlazione e a un grafico equivalente al diagramma di Venn dei casi corretti solo da ciascun componente.

In [ ]:
def save_figure(filename):
    plt.tight_layout(); plt.savefig(FIGURES_DIR / filename, dpi=200, bbox_inches="tight"); plt.show(); plt.close()
test_y = y_all[test_mask]
plt.figure(figsize=(6, 5))
for name in ("ensemble_uniform", "ensemble_weighted", "convnext"):
    fpr, tpr, _ = roc_curve(test_y, evaluation_probabilities[name][test_mask])
    plt.plot(fpr, tpr, label=f"{name} AUC={roc_auc_score(test_y, evaluation_probabilities[name][test_mask]):.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC — test"); plt.grid(alpha=0.3); plt.legend()
save_figure("roc_curve.png")

plt.figure(figsize=(6, 5))
for name in ("ensemble_uniform", "ensemble_weighted", "convnext"):
    precision, recall, _ = precision_recall_curve(test_y, evaluation_probabilities[name][test_mask])
    plt.plot(recall, precision, label=f"{name} AP={average_precision_score(test_y, evaluation_probabilities[name][test_mask]):.3f}")
plt.xlabel("Recall fake"); plt.ylabel("Precision fake"); plt.title("Precision-recall — test"); plt.grid(alpha=0.3); plt.legend()
save_figure("precision_recall_curve.png")

observed, predicted = calibration_curve(test_y, weighted_probability[test_mask], n_bins=10, strategy="quantile")
plt.figure(figsize=(6, 5)); plt.plot(predicted, observed, marker="o", label="Ensemble pesato"); plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("Probabilità predetta"); plt.ylabel("Frazione fake osservata"); plt.title("Calibrazione — test"); plt.grid(alpha=0.3); plt.legend()
save_figure("calibration_curve.png")

for matrix, filename, title in ((probability_correlation, "probability_correlation.png", "Correlazione probabilità — test"), (error_correlation, "error_correlation.png", "Correlazione errori — test")):
    plt.figure(figsize=(5, 4)); plt.imshow(matrix, vmin=-1, vmax=1, cmap="coolwarm"); plt.xticks(range(3), component_names, rotation=25, ha="right"); plt.yticks(range(3), component_names); plt.title(title)
    for row in range(3):
        for column in range(3):
            plt.text(column, row, f"{matrix.iloc[row, column]:.2f}", ha="center", va="center")
    plt.colorbar()
    save_figure(filename)

plt.figure(figsize=(6, 4)); plt.bar(list(unique_correct), list(unique_correct.values()), color=["#4C78A8", "#F58518", "#54A24B"])
plt.title("Casi test corretti esclusivamente da un componente"); plt.ylabel("Numero di casi"); plt.grid(axis="y", alpha=0.3)
save_figure("unique_correct_cases.png")

## 10. Intervalli bootstrap e confronto appaiato con ConvNeXt

Questa cella crea intervalli bootstrap al 95% per le metriche test, intervalli appaiati della differenza tra ensemble pesato e ConvNeXt e il test esatto di McNemar. Le conclusioni sono lasciate ai risultati: il notebook non dichiara superiorità senza evidenza statistica.

In [ ]:
def bootstrap_metric(targets, probabilities, metric, generator, repetitions=BOOTSTRAP_SAMPLES):
    values = []
    for _ in range(repetitions):
        indices = generator.integers(0, len(targets), len(targets))
        try:
            values.append(metric(targets[indices], probabilities[indices]))
        except ValueError:
            continue
    return np.percentile(values, [2.5, 97.5])
def paired_difference(targets, probabilities_a, probabilities_b, metric, generator, repetitions=BOOTSTRAP_SAMPLES):
    values = []
    for _ in range(repetitions):
        indices = generator.integers(0, len(targets), len(targets))
        try:
            values.append(metric(targets[indices], probabilities_a[indices]) - metric(targets[indices], probabilities_b[indices]))
        except ValueError:
            continue
    point = metric(targets, probabilities_a) - metric(targets, probabilities_b)
    lower, upper = np.percentile(values, [2.5, 97.5])
    return point, lower, upper

generator = np.random.default_rng(SEED)
bootstrap_rows = []
for name, probabilities in evaluation_probabilities.items():
    for metric_name, metric in (("roc_auc", roc_auc_score), ("pr_auc", average_precision_score)):
        lower, upper = bootstrap_metric(test_y, probabilities[test_mask], metric, generator)
        bootstrap_rows.append({"model": name, "metric": metric_name, "point_estimate": metric(test_y, probabilities[test_mask]), "ci_lower": lower, "ci_upper": upper})
    threshold = evaluation_thresholds[name]
    metric = lambda targets, probabilities: balanced_accuracy_score(targets, probabilities >= threshold)
    lower, upper = bootstrap_metric(test_y, probabilities[test_mask], metric, generator)
    bootstrap_rows.append({"model": name, "metric": "balanced_accuracy", "point_estimate": metric(test_y, probabilities[test_mask]), "ci_lower": lower, "ci_upper": upper})
paired_rows = []
for metric_name, metric in (("balanced_accuracy", lambda targets, probabilities: balanced_accuracy_score(targets, probabilities >= WEIGHTED_THRESHOLD)), ("roc_auc", roc_auc_score), ("pr_auc", average_precision_score)):
    point, lower, upper = paired_difference(test_y, weighted_probability[test_mask], evaluation_probabilities["convnext"][test_mask], metric, generator)
    paired_rows.append({"comparison": "ensemble_weighted_minus_convnext", "metric": metric_name, "point_estimate": point, "ci_lower": lower, "ci_upper": upper})

ensemble_correct = (weighted_probability[test_mask] >= WEIGHTED_THRESHOLD) == test_y
convnext_correct = (evaluation_probabilities["convnext"][test_mask] >= evaluation_thresholds["convnext"]) == test_y
ensemble_only, convnext_only = int((ensemble_correct & ~convnext_correct).sum()), int((~ensemble_correct & convnext_correct).sum())
mcnemar_p = binomtest(min(ensemble_only, convnext_only), n=ensemble_only + convnext_only, p=0.5, alternative="two-sided").pvalue if ensemble_only + convnext_only else 1.0
bootstrap_df, paired_bootstrap_df = pd.DataFrame(bootstrap_rows), pd.DataFrame(paired_rows)
bootstrap_df.to_csv(RUN_DIR / "bootstrap_confidence_intervals.csv", index=False)
paired_bootstrap_df.to_csv(RUN_DIR / "paired_bootstrap.csv", index=False)
pd.DataFrame([{"comparison": "ensemble_weighted_vs_convnext", "ensemble_only_correct": ensemble_only, "convnext_only_correct": convnext_only, "p_value": mcnemar_p}]).to_csv(RUN_DIR / "mcnemar_test.csv", index=False)
display(paired_bootstrap_df.round(4))

## 11. Bundle portabile e tabella finale

Questa cella copia gli artefatti base senza riaddestrare modelli, estrae il calibratore XGBoost, salva pesi e soglia appresi e produce un README per l'inferenza. Salva anche una tabella finale con metriche, tempi, dimensioni e hardware riportato dagli artefatti.

In [ ]:
BUNDLE_DIR = RUN_DIR / "bundle"; BUNDLE_DIR.mkdir(parents=True, exist_ok=False)
shutil.copy2(component_dirs["svm_rbf"] / "best_model.joblib", BUNDLE_DIR / "svm_pipeline.joblib")
shutil.copy2(component_dirs["xgboost"] / "best_model.json", BUNDLE_DIR / "xgboost_model.json")
joblib.dump(xgb_bundle["calibration"]["calibrator"], BUNDLE_DIR / "xgboost_calibrator.joblib")
shutil.copy2(component_dirs["cnn_spatial_head"] / "best_model_scripted.pt", BUNDLE_DIR / "cnn_spatial_scripted.pt")
ensemble_config = {
    "components": ["svm_rbf", "xgboost", "cnn_spatial_head"],
    "weights": {"svm_rbf": float(WEIGHTED_WEIGHTS[0]), "xgboost": float(WEIGHTED_WEIGHTS[1]), "cnn_spatial_head": float(WEIGHTED_WEIGHTS[2])},
    "threshold": float(WEIGHTED_THRESHOLD), "positive_class": POSITIVE_CLASS, "positive_label": POSITIVE_LABEL,
    "pooling": "mixed_by_component", "component_preprocessing": {"svm_rbf": svm_bundle["pooling"], "xgboost": xgb_bundle["pooling"], "cnn_spatial_head": "spatial_channel_normalized"},
    "cnn_temperature": CNN_TEMPERATURE, "split_hash": SPLIT_HASH, "run_ids": {name: path.name for name, path in component_dirs.items()}, "versions": VERSIONS,
}
with (BUNDLE_DIR / "ensemble_config.json").open("w", encoding="utf-8") as handle:
    json.dump(ensemble_config, handle, ensure_ascii=False, indent=2)
(BUNDLE_DIR / "README.md").write_text("Bundle dell'ensemble SVM RBF + XGBoost + CNN spatial head. Applicare i preprocessing dichiarati in ensemble_config.json, ottenere probabilità calibrate dei tre componenti, combinare con i pesi salvati e applicare la soglia salvata. La classe positiva è fake, label 1. ConvNeXt non è un componente del bundle.", encoding="utf-8")
if not all((BUNDLE_DIR / name).is_file() for name in ("svm_pipeline.joblib", "xgboost_model.json", "xgboost_calibrator.joblib", "cnn_spatial_scripted.pt", "ensemble_config.json", "README.md")):
    raise RuntimeError("Bundle incompleto")
if json.loads((BUNDLE_DIR / "ensemble_config.json").read_text(encoding="utf-8"))["split_hash"] != SPLIT_HASH:
    raise RuntimeError("Bundle e split non coincidono")

test_summary = metrics_df.loc[metrics_df["split"] == "test"].copy()
artifact_paths = {"svm_rbf": component_dirs["svm_rbf"] / "best_model.joblib", "xgboost": component_dirs["xgboost"] / "best_model.json", "cnn_spatial_head": component_dirs["cnn_spatial_head"] / "best_model_scripted.pt", "convnext": component_dirs["convnext"] / "best_model_scripted.pt"}
test_summary["model_size_bytes"] = test_summary["model"].map(lambda name: artifact_paths[name].stat().st_size if name in artifact_paths else np.nan)
test_summary["hardware"] = test_summary["model"].map(lambda name: str(metadata[name].get("versions", {}).get("device", "non registrato")) if name in metadata else "combinazione di modelli")
test_summary.to_csv(RUN_DIR / "model_comparison.csv", index=False)
display(test_summary.round(4))
print("Bundle verificato:", BUNDLE_DIR)

## Output e artefatti prodotti

Ogni esecuzione completa crea results/ensemble seguito dal run_id con:

- metrics.csv, model_comparison.csv e due file di predizioni standardizzate per ensemble uniforme e pesato;
- probability_correlation.csv, error_correlation.csv, agreement.csv e unique_correct_cases.csv;
- bootstrap_confidence_intervals.csv, paired_bootstrap.csv e mcnemar_test.csv;
- figures con ROC, precision-recall, calibrazione, correlazioni e casi corretti esclusivamente;
- bundle contenente svm_pipeline.joblib, xgboost_model.json, xgboost_calibrator.joblib, cnn_spatial_scripted.pt, ensemble_config.json e README.md.

Il notebook non riaddestra i componenti base e non include MLP o ConvNeXt nel voto. Pesi, temperature e soglie sono scelti esclusivamente sul validation; il test è riservato alla valutazione finale e al confronto statistico.